In [1]:
import torch
import torch.nn.functional as F
import math

torch.manual_seed(42)

# ------------------------------------------------------------
# 1. Toy sentence
# ------------------------------------------------------------

sentence = "i am driving"
words = sentence.split()

vocab = sorted(set(words))
stoi = {word: i for i, word in enumerate(vocab)}
itos = {i: word for word, i in stoi.items()}

x_ids = torch.tensor([stoi[word] for word in words])

print("Vocabulary:", stoi)
print("Input ids:", x_ids)

# ------------------------------------------------------------
# 2. Manual embedding table
# ------------------------------------------------------------

vocab_size = len(vocab)
embedding_dim = 4

E = torch.randn(vocab_size, embedding_dim)

# X shape: [T, embedding_dim]
X = E[x_ids]

print("\nX shape:", X.shape)
print("X:")
print(X)

# ------------------------------------------------------------
# 3. Self-attention parameters
# ------------------------------------------------------------

d_model = embedding_dim
d_k = 4
d_v = 4

W_Q = torch.randn(d_model, d_k) * 0.1
W_K = torch.randn(d_model, d_k) * 0.1
W_V = torch.randn(d_model, d_v) * 0.1

# ------------------------------------------------------------
# 4. Create Q, K, V from the SAME input sequence X
# ------------------------------------------------------------

Q = X @ W_Q   # [T, d_k]
K = X @ W_K   # [T, d_k]
V = X @ W_V   # [T, d_v]

print("\nQ shape:", Q.shape)
print("K shape:", K.shape)
print("V shape:", V.shape)

# ------------------------------------------------------------
# 5. Attention scores
# ------------------------------------------------------------

# Q @ K.T:
# [T, d_k] @ [d_k, T] = [T, T]
scores = Q @ K.T

# Scale by sqrt(d_k)
scores = scores / math.sqrt(d_k)

print("\nRaw attention scores shape:", scores.shape)
print(scores)

# ------------------------------------------------------------
# 6. Attention weights
# ------------------------------------------------------------

# softmax row-wise:
# each token decides how much to attend to every token
attention_weights = F.softmax(scores, dim=1)

print("\nAttention weights:")
print(attention_weights)

# ------------------------------------------------------------
# 7. Weighted sum of values
# ------------------------------------------------------------

# [T, T] @ [T, d_v] = [T, d_v]
output = attention_weights @ V

print("\nSelf-attention output shape:", output.shape)
print(output)

# ------------------------------------------------------------
# 8. Print readable attention table
# ------------------------------------------------------------

print("\nAttention table:")
header = f"{'from/to':>10s}"
for word in words:
    header += f"{word:>12s}"
print(header)

for i, from_word in enumerate(words):
    row = f"{from_word:>10s}"
    for weight in attention_weights[i]:
        row += f"{float(weight):12.3f}"
    print(row)

Vocabulary: {'am': 0, 'driving': 1, 'i': 2}
Input ids: tensor([2, 0, 1])

X shape: torch.Size([3, 4])
X:
tensor([[ 0.4617,  0.2674,  0.5349,  0.8094],
        [ 0.3367,  0.1288,  0.2345,  0.2303],
        [-1.1229, -0.1863,  2.2082, -0.6380]])

Q shape: torch.Size([3, 4])
K shape: torch.Size([3, 4])
V shape: torch.Size([3, 4])

Raw attention scores shape: torch.Size([3, 3])
tensor([[ 0.0001, -0.0018,  0.0140],
        [ 0.0002, -0.0008,  0.0027],
        [-0.0096,  0.0018, -0.0617]])

Attention weights:
tensor([[0.3320, 0.3314, 0.3366],
        [0.3332, 0.3328, 0.3340],
        [0.3378, 0.3416, 0.3206]])

Self-attention output shape: torch.Size([3, 4])
tensor([[-0.0928,  0.0896, -0.1243,  0.0386],
        [-0.0921,  0.0891, -0.1231,  0.0381],
        [-0.0890,  0.0864, -0.1167,  0.0358]])

Attention table:
   from/to           i          am     driving
         i       0.332       0.331       0.337
        am       0.333       0.333       0.334
   driving       0.338       0.342       